In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
df = pd.read_csv("../data/donors.csv")

print("Shape:", df.shape)
df.head()

## Классификация признаков

### Числовые
- family_size
- size_bytes

Стратегия: пропуски заполняются медианой, затем применяется StandardScaler.

### Категориальные
- author
- license
- behaviour_family
- family_evidence
- payload_included
- payload_format

Стратегия: пропуски заполняются отдельной категорией `missing`, затем применяется OneHotEncoder.

### Исключённые признаки
- donor_id — идентификатор
- source_url — URL
- main_py_sha256 — hash
- payload_b64 — закодированный payload
- verified — не используется в текущем пайплайне

|

In [ ]:
numeric_features = [
    "family_size",
    "size_bytes"
]

categorical_features = [
    "author",
    "license",
    "behaviour_family",
    "family_evidence",
    "payload_included",
    "payload_format"
]

features = numeric_features + categorical_features

X = df[features]

X_train, X_test = train_test_split(
    X,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [ ]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="missing"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)

preprocessor

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Original train:", X_train.shape)
print("Processed train:", X_train_processed.shape)

print("Original test:", X_test.shape)
print("Processed test:", X_test_processed.shape)

## Обоснование обработки пропусков

1. Для числовых признаков используется медиана. Медиана устойчива к выбросам и подходит для числовых данных.

2. Для категориальных признаков пропущенные значения заменяются отдельной категорией `missing`. Это позволяет сохранить информацию о наличии пропуска.

3. После заполнения числовые признаки масштабируются с помощью StandardScaler.

4. Категориальные признаки преобразуются с помощью OneHotEncoder.

5. Для предотвращения утечки данных ColumnTransformer обучается только на `X_train`.

6. Для `X_test` используется только `transform()`, без повторного обучения преобразований.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/medical_insurance.csv")

print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
import pandas as pd

df = pd.read_csv("../data/medical_insurance.csv")

print(df.shape)
print(df.columns.tolist())
df.head()